# Graph Neural Network — Relational Fraud Detection
> **Research question:** Do transactions sharing identifiers (same card, email,
> device) have correlated fraud patterns that a GNN can exploit?

**Core hypothesis:** Fraudsters reuse payment instruments. If card `X` was used
fraudulently at time T, a transaction at T+1 with the same card is more likely
to also be fraudulent. A GNN can propagate these signals across connected nodes.

**Architecture summary**
- Node = transaction; edge = shared identifier (card, address, email, device)
- Node features: numeric projection + categorical embeddings
- 3-layer residual GNN with mean aggregation + LayerNorm
- Node-level binary classification → fraud probability

**Graph construction:** Identity-edge-only (no k-NN).
Transactions sharing `card1`, `card4`, `card6`, `addr1`, `P_emaildomain`,
`R_emaildomain`, `id_30`, `id_31`, or `DeviceInfo` are connected.
- Small groups (≤10): full clique
- Larger groups: hub + chain pattern

In [ ]:
# Install required packages (run once in Colab)
!pip install -q torch pandas numpy scikit-learn pyarrow lightgbm

## 1 · Data Configuration

In [ ]:
import os

# ── Kaggle API download ───────────────────────────────────────────────────────
# One-time setup:
#   1. kaggle.com → Settings → API → "Create New Token"
#   2. Copy the entire token string Kaggle gives you (looks like {"username":...,"key":...})
#   3. In Colab: click the 🔑 Secrets icon → "Add new secret"
#        Name:  KAGGLE_API_TOKEN
#        Value: paste the full token string
#        Toggle "Notebook access" ON
#   4. Also accept the competition rules at kaggle.com/c/ieee-fraud-detection

from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

DATA_DIR = "data"
os.makedirs(f"{DATA_DIR}/raw",     exist_ok=True)
os.makedirs(f"{DATA_DIR}/interim", exist_ok=True)

print("Downloading IEEE-CIS Fraud Detection dataset from Kaggle …")
os.system("pip install -q kaggle")
os.system(f"kaggle competitions download -c ieee-fraud-detection -p {DATA_DIR}/raw/ --unzip -q")
print("Download complete. Files in data/raw/:")
os.system(f"ls -lh {DATA_DIR}/raw/")

## 2 · Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

# GNN uses full-graph training — the entire 590k-node graph is passed in one
# forward call. On Colab T4/A100, CUDA should have enough VRAM (~4-8 GB needed).
# Falls back to CPU if CUDA is unavailable (e.g. running locally on Apple Silicon).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3 · Load & Merge Data

In [ ]:
import os
import pandas as pd

def load_merged_train(data_dir="data"):
    """
    Load and merge the IEEE-CIS training files.

    Reads train_transaction.csv and train_identity.csv from data_dir/raw/,
    left-joins them on TransactionID so every transaction is kept (identity
    features are NaN for ~76% of rows that have no identity record), then
    caches the result to data_dir/interim/merged_train.parquet for fast
    re-loads.

    Returns
    -------
    pd.DataFrame  shape ~ (590540, 434)
    """
    cache = os.path.join(data_dir, "interim", "merged_train.parquet")
    if os.path.exists(cache):
        print(f"Loading from cache: {cache}")
        df = pd.read_parquet(cache)
        print(f"Shape: {df.shape}")
        return df

    raw = os.path.join(data_dir, "raw")
    print("Reading train_transaction.csv …")
    trans = pd.read_csv(os.path.join(raw, "train_transaction.csv"))
    print("Reading train_identity.csv …")
    ident = pd.read_csv(os.path.join(raw, "train_identity.csv"))
    print("Merging on TransactionID …")
    df = trans.merge(ident, how="left", on="TransactionID")
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    df.to_parquet(cache, index=False)
    print(f"Cached to {cache}.  Shape: {df.shape}")
    return df

df = load_merged_train(DATA_DIR)
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(f"Sorted by TransactionDT.  Shape: {df.shape}")

## 4 · Temporal Train / Val / Test Split
**70 / 15 / 15** split respecting `TransactionDT` order.
Defined before feature selection so LightGBM is fit on training data only.

In [ ]:
# Temporal 70 / 15 / 15 split — respects transaction time ordering.
# Using random splits on time-series data leaks future info into training.
n        = len(df)
n_train  = int(0.70 * n)
n_val    = int(0.15 * n)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_train + n_val))
test_idx  = list(range(n_train + n_val, n))
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")

## 5 · Feature Selection via LightGBM Importance
Same importance-ranked selection used by TabTransformer — both models see
identical features for a fair comparison.

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

def select_features_by_importance(df, train_idx, target_col="isFraud",
                                   max_numeric=50, max_categorical=20):
    """
    Rank all candidate features by LightGBM split importance, then pick the top N.

    Why this beats heuristic selection:
      The IEEE-CIS dataset has 339 anonymised V-columns — their names reveal
      nothing about their value. Heuristic selection (first 50 columns) grabs
      V1-V11, which may be less informative than V45 or V258. LightGBM quickly
      identifies which features the trees actually use for splits.

    Process:
      1. Drop columns that are >95% missing.
      2. Classify remaining columns as numeric or categorical.
      3. Fit a fast LightGBM (100 trees) on the TRAINING split only.
      4. Rank features by cumulative split gain.
      5. Return top max_numeric numeric + top max_categorical categorical.
    """
    ignore = {target_col, "TransactionID"}
    na_frac = df.isna().mean()
    kept = [c for c in na_frac[na_frac < 0.95].index if c not in ignore]

    num_cands, cat_cands = [], []
    for col in kept:
        dt = df[col].dtype
        if dt == "object" or str(dt) == "category":
            cat_cands.append(col)
        elif np.issubdtype(dt, np.integer):
            (cat_cands if df[col].nunique(dropna=True) <= 20 else num_cands).append(col)
        elif np.issubdtype(dt, np.floating):
            num_cands.append(col)

    all_cands = num_cands + cat_cands
    X = df[all_cands].copy()
    for col in cat_cands:
        X[col] = X[col].astype("category").cat.codes  # -1 for NaN, fine for trees
    X = X.fillna(-999).values.astype(np.float32)
    y = df[target_col].values

    pos = (y[train_idx] == 1).sum()
    neg = (y[train_idx] == 0).sum()
    print("Running LightGBM for feature importance (100 trees) …")
    lgb = LGBMClassifier(n_estimators=100, n_jobs=-1, verbose=-1,
                          random_state=42, scale_pos_weight=neg/pos)
    lgb.fit(X[train_idx], y[train_idx])

    importance = pd.Series(lgb.feature_importances_, index=all_cands).sort_values(ascending=False)
    num_set, cat_set = set(num_cands), set(cat_cands)
    numeric_cols     = [c for c in importance.index if c in num_set][:max_numeric]
    categorical_cols = [c for c in importance.index if c in cat_set][:max_categorical]

    print(f"Selected {len(numeric_cols)} numeric + {len(categorical_cols)} categorical by importance")
    print("Top 10 numeric:   ", numeric_cols[:10])
    print("Categorical:      ", categorical_cols)
    return numeric_cols, categorical_cols

numeric_cols, categorical_cols = select_features_by_importance(
    df, train_idx, target_col="isFraud", max_numeric=50, max_categorical=20
)

## 6 · Graph Construction

Builds the `edge_index` tensor (shape `[2, E]`) encoding which transactions
are connected.

**Why identity edges?**
k-NN edges (connecting numerically similar transactions) were tried but removed —
computing nearest neighbours on 590k × 50 features is O(N²) and takes hours.
Identity edges are O(N) and directly encode the fraud hypothesis.

In [ ]:
def build_transaction_graph(df, min_group_size=2, max_group_size=1000,
                             small_group_full_connect=10):
    """
    Build the transaction graph from shared identifiers.

    For each identity column (card1, addr1, email, device, etc.):
      - Group transactions that share the same value.
      - Skip NaN groups (transactions sharing only "unknown" are not related).
      - Small groups (≤ small_group_full_connect): fully connect (clique).
      - Larger groups: hub-and-spoke + sequential chain to limit edge count.

    Self-loops are added so each node can attend to its own features during
    message passing.

    Returns
    -------
    edge_index : LongTensor  shape (2, E)
    """
    df = df.reset_index(drop=True)
    src_list, dst_list = [], []

    id_cols = ["card1","card4","card6","addr1","P_emaildomain",
               "R_emaildomain","id_30","id_31","DeviceInfo"]
    active_cols = [c for c in id_cols if c in df.columns]

    for col in active_cols:
        vals   = df[col].astype(str)
        groups = vals.groupby(vals).groups
        for val, idxs in groups.items():
            if val in ("nan","None",""): continue
            grp  = sorted(list(idxs))
            size = len(grp)
            if size < min_group_size or size > max_group_size: continue

            if size <= small_group_full_connect:
                # Full clique — every pair connected
                for i in range(size):
                    for j in range(i + 1, size):
                        src_list += [grp[i], grp[j]]
                        dst_list += [grp[j], grp[i]]
            else:
                # Hub + chain pattern — keeps edge count linear in group size
                hub = grp[0]
                for other in grp[1:]:
                    src_list += [hub, other]; dst_list += [other, hub]
                for i in range(size - 1):
                    src_list += [grp[i], grp[i+1]]; dst_list += [grp[i+1], grp[i]]

    if not src_list:
        return torch.empty((2, 0), dtype=torch.long)

    ei = torch.tensor([src_list, dst_list], dtype=torch.long)
    ei = torch.unique(ei.t(), dim=0).t().contiguous()   # deduplicate

    # Self-loops: each node aggregates its own features
    n   = len(df)
    sl  = torch.arange(n, dtype=torch.long)
    ei  = torch.cat([ei, torch.stack([sl, sl])], dim=1)
    return ei

print("Building transaction graph …")
edge_index = build_transaction_graph(df)
print(f"edge_index shape: {edge_index.shape}  ({edge_index.shape[1]:,} edges)")

## 7 · Model Architecture — SimpleGNN

A 3-layer residual GNN with mean aggregation.

**Message passing (mean aggregation):**
For each node `v`, the new representation is the mean of its neighbours' features:
`h_v' = mean({ h_u : u ∈ N(v) })`
Then: `h_v = ReLU(W · h_v') + h_v` (residual connection preserves local info)

In [ ]:
class SimpleGNN(nn.Module):
    """
    3-layer residual GNN for node-level fraud classification.

    Parameters
    ----------
    num_numeric : int
        Number of numeric input features per node.
    num_categories_per_col : list[int]
        Vocabulary sizes for each categorical column.
    embed_dim : int
        Embedding dimension for numeric projection and categorical embeddings.
    hidden_dim : int
        Hidden dimension used throughout the GNN layers.
    dropout : float
        Dropout probability (applied after each GNN layer).
    """

    def __init__(self, num_numeric, num_categories_per_col,
                 embed_dim=32, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(size + 1, embed_dim) for size in num_categories_per_col
        ])
        self.num_linear   = nn.Linear(num_numeric, embed_dim)
        total_in          = embed_dim * (1 + len(num_categories_per_col))
        self.input_linear = nn.Linear(total_in, hidden_dim)

        # Three residual GNN layers
        self.gcn1, self.gcn2, self.gcn3 = (
            nn.Linear(hidden_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.act       = nn.ReLU()
        self.out_linear = nn.Linear(hidden_dim, 1)

    def _mean_agg(self, x, edge_index):
        """Mean-aggregate neighbour features for each node."""
        if edge_index.numel() == 0: return x
        src, dst = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        deg = torch.zeros(x.size(0), device=x.device).index_add_(
            0, dst, torch.ones(len(dst), device=x.device)
        ).clamp_min(1.0).unsqueeze(1)
        return agg / deg

    def forward(self, x_num, x_cat, edge_index):
        # Encode features into a shared embedding space
        h = torch.cat(
            [self.act(self.num_linear(x_num))] +
            [emb(x_cat[:, i].clamp(0, emb.num_embeddings - 1))
             for i, emb in enumerate(self.cat_embeddings)],
            dim=1,
        )
        h = self.act(self.input_linear(h))

        # Three residual message-passing layers
        for gcn, norm in [(self.gcn1, self.norm1),
                          (self.gcn2, self.norm2),
                          (self.gcn3, self.norm3)]:
            h_res = h
            h     = self.dropout(self.act(gcn(self._mean_agg(h, edge_index))))
            h     = norm(h + h_res)

        return self.out_linear(h).squeeze(-1)

## 8 · Training Function

Key differences from TabTransformer training:
- **Full-graph training** — the entire graph is passed to the model in one call
  (no mini-batching; the GNN needs global message-passing context)
- **Early stopping with patience=20** — GNN converges more slowly than mini-batch models
- **150–250 epochs** recommended; the model was still improving at epoch 150

In [ ]:
def train_gnn(df, numeric_cols, categorical_cols, target_col="isFraud",
              num_epochs=250, lr=1e-3, train_idx=None, val_idx=None,
              test_idx=None, early_stopping_patience=20):
    """
    Full-graph training of the SimpleGNN.

    The entire graph (all 590k nodes + edges) is used in every forward pass.
    This is memory-intensive but necessary because GNN message passing needs
    global context — mini-batching would require graph partitioning (not implemented).

    Returns
    -------
    metrics : dict
    model   : SimpleGNN (best checkpoint)
    """
    cols = numeric_cols + categorical_cols + [target_col]
    df   = df[cols].copy().reset_index(drop=True)

    # ── Numeric scaling ───────────────────────────────────────────────────────
    num_df = df[numeric_cols].fillna(0.0).astype("float32")
    num_df = (num_df - num_df.mean()) / num_df.std().replace(0, 1.0)
    df[numeric_cols] = num_df
    x_num = torch.tensor(num_df.values, dtype=torch.float32)

    # ── Categorical encoding ──────────────────────────────────────────────────
    cat_sizes, cat_arrays = [], []
    for col in categorical_cols:
        vals    = df[col].astype(str)
        mapping = {v: i for i, v in enumerate(sorted(vals.unique()))}
        cat_sizes.append(len(mapping))
        cat_arrays.append(vals.map(mapping).astype("int64").values)

    x_cat = (torch.tensor(np.stack(cat_arrays, axis=1), dtype=torch.long)
             if cat_arrays else torch.empty((len(df), 0), dtype=torch.long))

    y = torch.tensor(df[target_col].values.astype("float32"), dtype=torch.float32)

    # ── Graph ─────────────────────────────────────────────────────────────────
    if "edge_index" not in dir():
        print("Building graph …")
    ei = build_transaction_graph(df)

    # ── Splits ────────────────────────────────────────────────────────────────
    if train_idx is None or val_idx is None:
        n_tr = int(0.8 * len(df))
        train_idx = list(range(0, n_tr)); val_idx = list(range(n_tr, len(df)))
    tr  = torch.tensor(train_idx, dtype=torch.long)
    val = torch.tensor(val_idx,   dtype=torch.long)
    tst = torch.tensor(test_idx,  dtype=torch.long) if test_idx else None

    # Move to device
    x_num, x_cat, y, ei = x_num.to(device), x_cat.to(device), y.to(device), ei.to(device)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = SimpleGNN(len(numeric_cols), cat_sizes).to(device)
    pos   = (y[tr] == 1).sum(); neg = (y[tr] == 0).sum()
    criterion = nn.BCEWithLogitsLoss(pos_weight=(neg / pos).clamp(min=1.0))
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_pr, best_epoch, pat_ctr, best_state = -1.0, 0, 0, None

    for epoch in range(1, num_epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(x_num, x_cat, ei)
        loss   = criterion(logits[tr], y[tr])
        if not torch.isfinite(loss): print(f"Non-finite loss at epoch {epoch}"); break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(x_num, x_cat, ei))[val].cpu().numpy()
        ep_pr = average_precision_score(y[val].cpu().numpy(), vp)
        print(f"[Epoch {epoch:3d}] loss={loss.item():.4f}  val_pr_auc={ep_pr:.4f}")

        if ep_pr > best_pr:
            best_pr, best_epoch, pat_ctr = ep_pr, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            pat_ctr += 1
            if pat_ctr >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch} (pr_auc={best_pr:.4f})")
                break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"Restored best model from epoch {best_epoch}")

    # ── Evaluation ────────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(x_num, x_cat, ei))

    def eval_split(idx_tensor, label):
        yt = y[idx_tensor].cpu().numpy()
        ys = probs[idx_tensor].cpu().numpy()
        precs, recs, thrs = precision_recall_curve(yt, ys)
        f1s = 2 * precs[:-1] * recs[:-1] / (precs[:-1] + recs[:-1] + 1e-8)
        thr = float(thrs[f1s.argmax()])
        yp  = (ys >= thr).astype("int32")
        return {
            f"{label}_precision": precision_score(yt, yp, zero_division=0),
            f"{label}_recall":    recall_score(yt, yp, zero_division=0),
            f"{label}_f1":        f1_score(yt, yp, zero_division=0),
            f"{label}_roc_auc":   roc_auc_score(yt, ys),
            f"{label}_pr_auc":    average_precision_score(yt, ys),
            f"{label}_threshold": thr,
        }

    metrics = eval_split(val, "val")
    print("\n===== GNN — Validation =====")
    for k, v in metrics.items(): print(f"  {k:22s}: {v:.4f}")

    if tst is not None:
        test_m = eval_split(tst, "test")
        print("\n===== GNN — Test =====")
        for k, v in test_m.items(): print(f"  {k:22s}: {v:.4f}")
        metrics.update(test_m)

    return metrics, model

## 9 · Run Training

In [ ]:
# NOTE: Building the graph takes ~2-3 minutes.
# Training is ~13 sec/epoch on CPU — early stopping will stop before epoch 250
# once val PR-AUC plateaus (typically around epoch 150-200).
print("Building graph …")
edge_index = build_transaction_graph(df)
print(f"Graph built: {edge_index.shape[1]:,} edges")

gnn_metrics, gnn_model = train_gnn(
    df, numeric_cols, categorical_cols,
    target_col="isFraud",
    num_epochs=250,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
)

## 10 · Results Summary

In [ ]:
print("\n" + "="*45)
print("  GNN — Final Results")
print("="*45)
for k, v in gnn_metrics.items():
    print(f"  {k:24s}: {v:.4f}")